# Harmonizome — API & bulk

Maayan-Lab's [Harmonizome](https://maayanlab.cloud/Harmonizome/) packages
~114 curated gene-attribute datasets (CCLE, GTEx, ENCODE, HPA, KEGG,
Reactome, …) under a single REST + bulk-TSV/GMT API.

| Mode | Functions |
|---|---|
| **REST** | `list_datasets`, `get_dataset_metadata` |
| **Bulk** | `download_datasets`, `get_gmt`, `load_gene_attribute_matrix` |


**Execution policy (2026-10-29):** code cells are reproducible offline examples
with synthetic data. Upstream requests are labeled **Live recipe — not
executed** and shown in Markdown; their outputs depend on the chosen source
release. Install optional source extras before running live recipes.


**Live recipe — not executed:**

```python
from biodb import harmonizome
```


## 1. REST — enumerate the catalog


**Live recipe — not executed:**

```python
catalog = harmonizome.list_datasets()
print(f"{len(catalog)} datasets")
catalog[:5]
```


Per-dataset metadata — author info, gene count, attribute count.


**Live recipe — not executed:**

```python
meta = harmonizome.get_dataset_metadata("GTEx Tissue Sample Gene Expression Profiles")
{k: meta.get(k) for k in ("name", "description", "geneCount", "attributeCount") if k in meta}
```


## 2. Bulk — GMT and gene-attribute matrices

`get_gmt(datasets=[dataset])` downloads + parses the GMT (gene set per
attribute). Cached under `~/.cache/biodb/harmonizome/`.


```python
# Live download: choose a dataset from the current catalog.
gmt = harmonizome.get_gmt(datasets=["Achilles Cell Line Gene Essentiality Profiles"])
print(gmt[["id", "label", "gene"]].head())

# The full (gene × attribute) matrix for a chosen dataset:
mat, tissue_metadata, column_metadata = harmonizome.load_gene_attribute_matrix(
    "CCLE Cell Line Gene Expression Profiles"
)
mat.shape
```

Some matrices are large (CCLE ~ 20 k × 1 k); start with a smaller
dataset like `"Achilles Cell Line Gene Essentiality Profiles"` to
learn the API shape before scaling up.


## Offline executed counterpart

This fixture exercises the public API on synthetic data.


In [1]:
import gzip
import tempfile
from pathlib import Path

import pandas as pd

work = Path(tempfile.mkdtemp(prefix="biodb-tutorial-"))

In [2]:
from biodb import harmonizome

name = "Synthetic Dataset"
directory = work / name
directory.mkdir()
body = "GeneSym\tNA\tGeneID\tcellA\tcellB\nGENEA\tNA\t1\t0.5\t1.0\n"
(directory / "gene_attribute_matrix.txt.gz").write_bytes(gzip.compress(body.encode()))
matrix, tissue_metadata, column_metadata = harmonizome.load_gene_attribute_matrix(
    name, cache_dir=work
)
print(matrix)
assert matrix.shape[0] == 1
# get_gmt accepts a previously built merged Parquet cache. Source metadata
# acquisition remains an explicit live recipe above.
pd.DataFrame(
    {"dataset": [name], "id": ["SET1"], "label": ["Synthetic"], "gene": ["GENEA"]}
).to_parquet(work / "gmt_merged.parquet", index=False)
print(harmonizome.get_gmt(cache_dir=work, verbose=0))

  GeneSym  GeneID  cellA  cellB
0   GENEA       1    0.5    1.0
             dataset    id      label   gene
0  Synthetic Dataset  SET1  Synthetic  GENEA
